In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy qiskit-ibm-catalog scikit-learn
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Singularity Machine Learning - Classification: Isang Qiskit Function ng Multiverse Computing
> **Note:** * Ang Qiskit Functions ay isang experimental na feature na available lamang sa mga gumagamit ng IBM Quantum&reg; Premium Plan, Flex Plan, at On-Prem (sa pamamagitan ng IBM Quantum Platform API) Plan. Nasa preview release status ito at maaaring magbago.

## Pangkalahatang-ideya
Sa pamamagitan ng function na "Singularity Machine Learning - Classification", maaari kang malutas ng mga tunay na problema sa machine learning sa quantum hardware nang hindi kailangan ng kaalaman sa quantum. Ang Application function na ito, batay sa ensemble methods, ay isang hybrid classifier. Ginagamit nito ang mga klasikal na pamamaraan tulad ng boosting, bagging, at stacking para sa unang ensemble training. Pagkatapos, ginagamit ang mga quantum algorithm tulad ng variational quantum eigensolver (VQE) at quantum approximate optimization algorithm (QAOA) para mapahusay ang diversity, generalization capabilities, at kabuuang complexity ng na-train na ensemble.

Hindi tulad ng ibang quantum machine learning na solusyon, kaya ng function na ito na hawakan ang malalaking dataset na may milyun-milyong halimbawa at feature nang hindi nililimitahan ng bilang ng mga qubit sa target QPU. Ang bilang ng mga qubit ay nagtatakda lamang ng laki ng ensemble na maaaring i-train. Napaka-flexible din nito, at maaaring gamitin para malutas ang mga problema sa classification sa iba't ibang larangan, kabilang ang finance, healthcare, at cybersecurity.
Palagi itong nakakamit ng mataas na accuracy sa mga klasikal na mahirap na problema na kinabibilangan ng high-dimensional, maingay, at imbalanced na dataset.
![Paano ito gumagana](../docs/images/guides/multiverse-computing-singularity/how_it_works.avif)
Ginawa ito para sa:
1. Mga engineer at data scientist sa mga kumpanya na naghahanap na palawakin ang kanilang tech offerings sa pamamagitan ng pagsasama ng quantum machine learning sa kanilang mga produkto at serbisyo,
2. Mga mananaliksik sa mga quantum research lab na nag-eeksplora ng mga aplikasyon sa quantum machine learning at gustong gamitin ang quantum computing para sa mga gawain sa classification, at
3. Mga estudyante at guro sa mga institusyong pang-edukasyon sa mga kurso tulad ng machine learning, at gustong ipakita ang mga kalamangan ng quantum computing.

Ipinapakita ng sumusunod na halimbawa ang iba't ibang functionality nito, kabilang ang `create`, `list`, `fit`, at `predict`, at ipinapakita ang paggamit nito sa isang synthetic na problema na binubuo ng dalawang magkabaluktot na kalahating bilog, isang notoriously challenging na problema dahil sa nonlinear na hangganan ng desisyon nito.


## Paglalarawan ng function
Ang Qiskit Function na ito ay nagbibigay-daan sa mga gumagamit na malutas ng mga problema sa binary classification gamit ang quantum-enhanced ensemble classifier ng Singularity. Sa likod ng eksena, gumagamit ito ng hybrid approach para klasikal na i-train ang isang ensemble ng mga classifier sa labeled na dataset, at pagkatapos ay i-optimize ito para sa maximum na diversity at generalization gamit ang Quantum Approximate Optimization Algorithm (QAOA) sa mga IBM&reg; QPU. Sa pamamagitan ng user-friendly na interface, maaaring i-configure ng mga gumagamit ang isang classifier ayon sa kanilang mga pangangailangan, i-train ito sa dataset na kanilang pinili, at gamitin ito para gumawa ng mga hula sa isang dataset na hindi pa nakikita noon.

Para malutas ang isang generic na problema sa classification:
1. I-preprocess ang dataset, at hatiin ito sa mga training at testing set. Opsyonal, maaari mo ring hatiin ang training set sa mga training at validation set. Maaari itong makamit gamit ang [scikit-learn](https://scikit-learn.org/1.5/modules/generated/sklearn.model_selection.train_test_split.html).
2. Kung ang training set ay imbalanced, maaari mo itong i-resample para balansehin ang mga klase gamit ang [imbalanced-learn](https://imbalanced-learn.org/stable/introduction.html#problem-statement-regarding-imbalanced-data-sets).
3. I-upload ang mga training, validation, at test set nang hiwalay sa storage ng function gamit ang `file_upload` na paraan ng catalog, na ipinasa ang kaugnay na path sa bawat pagkakataon.
4. I-initialize ang quantum classifier sa pamamagitan ng paggamit ng `create` action ng function, na tumatanggap ng mga hyperparameter tulad ng bilang at uri ng mga learner, ang regularization (lambda value), at mga opsyon sa optimization kabilang ang bilang ng mga layer, uri ng klasikal na optimizer, ang quantum backend, at iba pa.
5. I-train ang quantum classifier sa training set gamit ang `fit` action ng function, ipinasa ang labeled na training set, at ang validation set kung naaangkop.
6. Gumawa ng mga hula sa dating hindi nakitang test set gamit ang `predict` action ng function.
## Action-based na pamamaraan
Gumagamit ang function ng action-based na pamamaraan. Maaari mo itong isipin bilang isang virtual na kapaligiran kung saan gumagamit ka ng mga action para magsagawa ng mga gawain o baguhin ang estado nito. Sa kasalukuyan, nag-aalok ito ng mga sumusunod na action: [list](#1-list), [create](#2-create), [delete](#3-delete), [fit](#4-fit), [predict](#5-predict), [fit_predict](#6-fit-predict), at [create_fit_predict](#7-create-fit-predict). Ipinapakita ng sumusunod na halimbawa ang `create_fit_predict` na action.

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load function
singularity = catalog.load("multiverse/singularity")

## Examples

### Classify a dataset

In this example, you'll use the "Singularity Machine Learning - Classification" function to classify a dataset consisting of two interleaving, moon-shaped half-circles. The dataset is synthetic, two-dimensional, and labeled with binary labels. It is created to be challenging for algorithms such as centroid-based clustering and linear classification.

![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)

Through this process, you'll learn how to create the classifier, fit it to the training data, use it to predict on the test data, and delete the classifier when you're finished.

Before starting, you need to install [scikit-learn](https://scikit-learn.org/). Install it using the following command:

```sh
python3 -m pip install scikit-learn
```

Perform the following steps:

1) Create the synthetic dataset using the [`make_moons`](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html) function from [scikit-learn](https://scikit-learn.org/).
2) Upload the generated synthetic dataset to the [shared data directory](https://qiskit.github.io/qiskit-serverless/function_features/experimental/manage_data_directory.html).
3) Create the quantum-enhanced classifier using the [`create`](/docs/api/functions/multiverse-computing-singularity#create) action.
4) Enlist your classifiers using the [`list`](/docs/api/functions/multiverse-computing-singularity#list) action.
5) Train the classifier on the train data using the [`fit`](/docs/api/functions/multiverse-computing-singularity#fit) action.
6) Use the trained classifier to predict on the test data using the [`predict`](/docs/api/functions/multiverse-computing-singularity#predict) action.
7) Delete the classifier using the [`delete`](/docs/api/functions/multiverse-computing-singularity#delete) action.
8) Clean up after you're done.

**Step 1.** Import the necessary modules and generate the synthetic dataset, then split it into training and test datasets.

In [2]:
# import the necessary modules for this example
import os
import tarfile
import numpy as np

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# generate the synthetic dataset
X, y = make_moons(n_samples=10000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# print the first 10 samples of the training dataset
print("Features:", X_train[:10, :])
print("Targets:", y_train[:10])

Features: [[ 0.84757037 -0.48831433]
 [ 0.98132552  0.19235443]
 [-0.71626723  0.6978261 ]
 [ 1.18957848 -0.48186557]
 [ 0.52118982 -0.37791846]
 [ 0.81115408  0.58483251]
 [ 0.48706462  0.87336593]
 [-0.81880144  0.57407682]
 [ 1.67335408 -0.23932015]
 [ 0.50181306  0.8649761 ]]
Targets: [1 0 0 1 1 0 0 0 1 0]


### 1. List

Kinukuha ng `list` action ang lahat ng nakaimbak na classifier sa format na `*.pkl.tar` mula sa shared data directory. Maaari mo ring ma-access ang mga nilalaman ng direktoryong ito sa pamamagitan ng paggamit ng paraan ng `catalog.files()`. Sa pangkalahatan, hinahanap ng list action ang mga file na may extension na `*.pkl.tar` sa shared data directory at ibinabalik ang mga ito sa format na list.
#### Mga input
|     Pangalan    |    Uri     | Paglalarawan |   Kinakailangan  |
|-------------|-------------|-------------|-------------|
| `action` | `str` | Ang pangalan ng action mula sa `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` at `delete`. | Oo |

#### Paggamit

In [3]:
def make_tarfile(file_path, tar_file_name):
    with tarfile.open(tar_file_name, "w") as tar:
        tar.add(file_path, arcname=os.path.basename(file_path))


# save the training and test datasets on your local disk
np.save("X_train.npy", X_train)
np.save("y_train.npy", y_train)
np.save("X_test.npy", X_test)
np.save("y_test.npy", y_test)

# create tar files for the datasets
make_tarfile("X_train.npy", "X_train.npy.tar")
make_tarfile("y_train.npy", "y_train.npy.tar")
make_tarfile("X_test.npy", "X_test.npy.tar")
make_tarfile("y_test.npy", "y_test.npy.tar")

# upload the datasets to the shared data directory
catalog.file_upload("X_train.npy.tar", singularity)
catalog.file_upload("y_train.npy.tar", singularity)
catalog.file_upload("X_test.npy.tar", singularity)
catalog.file_upload("y_test.npy.tar", singularity)

# view/enlist the uploaded files in the shared data directory
print(catalog.files(singularity))

['X_test.npy.tar', 'X_train.npy.tar', 'y_test.npy.tar', 'y_train.npy.tar']


### 2. Create

Ang `create` action ay gumagawa ng classifier ng tinukoy na uri ng `quantum_classifier` sa pamamagitan ng paggamit ng mga ibinigay na parameter, at sine-save ito sa shared data directory.

> **Note:** Ang function ay kasalukuyang sumusuporta lamang sa `QuantumEnhancedEnsembleClassifier`.
#### Mga input
|     Pangalan    |    Uri     | Paglalarawan | Kinakailangan  | Default |
|-------------|-------------|-------------|-----------|---------|
| `action` | `str` | Ang pangalan ng action mula sa `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` at `delete`. | Oo | - |
| `name` | `str` | Ang pangalan ng quantum classifier, hal., `spam_classifier`. | Oo | - |
| `instance` | `str` | IBM instance. | Oo | - |
| `backend_name` | `str` | IBM compute resource. Ang default ay `None`, ibig sabihin ay gagamitin ang backend na may pinakamaunting pending na trabaho. | Hindi | `None` |
| `quantum_classifier` | `str` | Ang uri ng quantum classifier, i.e., `QuantumEnhancedEnsembleClassifier`. | Hindi | `QuantumEnhancedEnsembleClassifier` |
| `num_learners` | `integer` | Ang bilang ng mga learner sa ensemble. | Hindi | `10` |
| `learners_types` | `list` | Mga uri ng learner. Kabilang sa mga sinusuportahang uri ang: `DecisionTreeClassifier`, `GaussianNB`, `KNeighborsClassifier`, `MLPClassifier`, at `LogisticRegression`. Ang karagdagang detalye para sa bawat isa ay makikita sa [dokumentasyon ng scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Hindi | `[DecisionTreeClassifier]` |
| `learners_proportions` | `list` | Mga proporsyon ng bawat uri ng learner sa ensemble. | Hindi | `[1.0]` |
| `learners_options` | `list` | Mga opsyon para sa bawat uri ng learner sa ensemble. Para sa kumpletong listahan ng mga opsyon na naaayon sa napiling uri ng learner, tingnan ang [dokumentasyon ng scikit-learn](https://scikit-learn.org/stable/supervised_learning.html). | Hindi | `[{"max_depth": 3, "splitter": "random", "class_weight": None}]` |
| `regularization_type` | `str` o `list` | Uri/Mga uri ng regularization na gagamitin: `onsite` o `alpha`. Ang `onsite` ay nagkokontrol sa onsite term kung saan ang mas mataas na halaga ay humahantong sa mas sparse na ensemble. Ang `alpha` ay nagkokontrol sa trade-off sa pagitan ng interaction at onsite na term kung saan ang mas mababang halaga ay humahantong sa mas sparse na ensemble. Kung ibinigay ang isang list, ang mga modelo ay ita-train para sa bawat uri at pipiliin ang pinakamahusay na gumaganap. | Hindi | `onsite` |
| `regularization` | `str` o `float` o `list` | Halaga ng regularization. Nakakulong sa pagitan ng `0` at `+inf` kung ang regularization_type ay `onsite`. Nakakulong sa pagitan ng `0` at `1` kung ang regularization_type ay `alpha`. Kung itinakda sa `auto`, ginagamit ang auto-regularization - ang pinakamainam na regularization parameter ay nahanap sa pamamagitan ng binary search na may nais na ratio ng mga piniling classifier sa kabuuang classifier (`regularization_desired_ratio`) at ang upper bound para sa regularization parameter (`regularization_upper_bound`). Kung ibinigay ang isang list, ang mga modelo ay ita-train para sa bawat halaga at pipiliin ang pinakamahusay na gumaganap. | Hindi | `0.01` |
| `regularization_desired_ratio` | `float` o `list` | Nais na ratio/Mga ratio ng mga piniling classifier sa kabuuang classifier para sa auto-regularization. Kung ibinigay ang isang list, ang mga modelo ay ita-train para sa bawat ratio at pipiliin ang pinakamahusay na gumaganap. | Hindi | `0.75` |
| `regularization_upper_bound` | `float` o `list` | Upper bound/Mga upper bound para sa regularization parameter kapag gumagamit ng auto-regularization. Kung ibinigay ang isang list, ang mga modelo ay ita-train para sa bawat upper bound at pipiliin ang pinakamahusay na gumaganap. | Hindi | `200` |
| `weight_update_method` | `str` | Pamamaraan para sa pag-update ng sample weights mula sa `logarithmic` at `quadratic`. | Hindi | `logarithmic` |
| `sample_scaling` | `boolean` | Kung dapat ilapat ang sample scaling. | Hindi | `False` |
| `prediction_scaling` | `float` | Scaling factor para sa mga hula. | Hindi | `None` |
| `optimizer_options` | `dictionary` | Mga opsyon ng QAOA optimizer. Ang listahan ng mga available na opsyon ay ipinakita sa bandang huli ng dokumentasyong ito. | Hindi | ... |
| `voting` | `str` | Gumamit ng majority voting (`hard`) o average ng mga probabilidad (`soft`) para sa pagsasama-sama ng mga hula/probabilidad ng mga learner. | Hindi | `hard` |
| `prob_threshold` | `float` | Pinakamainam na threshold ng probabilidad. | Hindi | `0.5` |
| `random_state` | `integer` | Kontrolin ang randomness para sa repeatability. | Hindi | `None` |


- Bukod dito, ang `optimizer_options` ay nakalista tulad ng sumusunod:

|     Pangalan    |    Uri     | Paglalarawan | Kinakailangan  | Default |
|-------------|-------------|-------------|-----------|---------|
| `num_solutions` | `integer` | Ang bilang ng mga solusyon | Hindi | `1024` |
| `reps` | `integer` | Ang bilang ng mga paulit-ulit | Hindi | `4` |
| `sparsify` | `float` | Ang sparsification threshold | Hindi | `0.001` |
| `theta` | `float` | Ang paunang halaga ng theta, isang variational parameter ng QAOA | Hindi | `None` |
| `simulator` | `boolean` | Kung gagamit ng simulator o QPU | Hindi | `False` |
| `classical_optimizer` | `str` | Pangalan ng klasikal na optimizer para sa QAOA. Lahat ng solver na inaalok ng SciPy, tulad ng nakalista [dito](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html#scipy.optimize.minimize), ay maaaring gamitin. Kakailanganin mong itakda ang `classical_optimizer_options` nang naaangkop | Hindi | `COBYLA` |
| `classical_optimizer_options` | `dictionary` | Mga opsyon ng klasikal na optimizer. Para sa kumpletong listahan ng mga available na opsyon, tingnan ang [dokumentasyon ng SciPy](https://docs.scipy.org/doc/scipy/reference/) | Hindi | `{"maxiter": 60}` |
| `optimization_level` | `integer` | Ang lalim ng QAOA circuit | Hindi | `3` |
| `num_transpiler_runs` | `integer` | Bilang ng mga run ng Transpiler | Hindi | `30` |
| `pass_manager_options` | `dictionary` | Mga opsyon para sa pagbuo ng preset pass manager | Hindi | `{"approximation_degree": 1.0}` |
| `estimator_options` | `dictionary` | Mga opsyon ng Estimator. Para sa kumpletong listahan ng mga available na opsyon, tingnan ang [dokumentasyon ng Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) | Hindi | `None` |
| `sampler_options` | `dictionary` | Mga opsyon ng Sampler. Para sa kumpletong listahan ng mga available na opsyon, tingnan ang [dokumentasyon ng Qiskit Runtime Client](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options) | Hindi | `None` |

- Ang default na `estimator_options` ay:

|     Pangalan    |    Uri     | Halaga  |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `2` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |
| `resilience_options` | `dictionary` | `{"zne_mitigation": False, "zne": {"amplifier": "pea", "noise_factors": [1.0, 1.3, 1.6], "extrapolator": ["linear", "polynomial_degree_2", "exponential"],}}` |

- Ang default na `sampler_options` ay:

|     Pangalan    |    Uri     | Halaga |
|-------------|-------------|-------------|
| `default_shots` | `integer` | `1024` |
| `resilience_level` | `integer` | `1` |
| `twirling` | `dictionary` | `{"enable_gates": True}` |
| `dynamical_decoupling` | `dictionary` | `{"enable": True}` |

#### Paggamit

In [4]:
job = singularity.run(
    action="create",
    name="my_classifier",
    num_learners=10,
    learners_types=[
        "DecisionTreeClassifier",
        "KNeighborsClassifier",
    ],
    learners_proportions=[0.5, 0.5],
    learners_options=[{}, {}],
    regularization=0.01,
    weight_update_method="logarithmic",
    sample_scaling=True,
    optimizer_options={"simulator": True},
    voting="soft",
    prob_threshold=0.5,
)

print(job.result())

{'status': 'ok', 'message': 'Classifier created.', 'data': {}, 'metadata': {'resource_usage': {}}}


In [5]:
# list available classifiers using the list action
job = singularity.run(action="list")

print(job.result())

# you can also find your classifiers in the shared data directory with a *.pkl.tar extension
print(catalog.files(singularity))

{'status': 'ok', 'message': 'Classifiers listed.', 'data': {'classifiers': ['my_classifier']}, 'metadata': {'resource_usage': {}}}


['X_test.npy.tar', 'X_train.npy.tar', 'my_classifier.pkl.tar', 'y_test.npy.tar', 'y_train.npy.tar']


#### Mga validation
- `name`:
    - Ang pangalan ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Ang isang classifier na may parehong pangalan ay dapat na mayroon na sa shared data directory.
### 4. Fit

Sine-train ng `fit` action ang isang classifier gamit ang ibinigay na training data.

#### Mga input
|     Pangalan    |    Uri     | Paglalarawan |   Kinakailangan  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Ang pangalan ng action. Dapat ay `fit`. | Oo |
| `name`      | `str`    | Ang pangalan ng classifier na ita-train. | Oo |
| `X`         | `array` o `list` o `str` | Ang training data. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `y`         | `array` o `list` o `str` | Ang mga target na halaga ng training. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `fit_params`| `dictionary`| Mga karagdagang parameter na ipapasa sa `fit` na paraan ng classifier. | Hindi |

##### fit_params
|     Pangalan    |    Uri     | Paglalarawan |   Kinakailangan  |   Default   |
|-------------|-------------|-------------|-------------|-------------|
| `validation_data` | `tuple` | Ang validation data at mga label. | Hindi | `None` |
| `pos_label` | `integer` o `str` | Ang class label na imi-map sa 1. | Hindi | `None` |
| `optimization_data` | `str` | Dataset para i-optimize ang ensemble. Maaaring isa sa: `train`, `validation`, `both`. | Hindi | `train` |

#### Paggamit

In [6]:
job = singularity.run(
    action="fit",
    name="my_classifier",
    X="X_train.npy",  # you do not need to specify the tar extension
    y="y_train.npy",  # you do not need to specify the tar extension
)

print(job.result())

{'status': 'ok', 'message': 'Classifier fitted.', 'data': {}, 'metadata': {'resource_usage': {'RUNNING: MAPPING': {'CPU_TIME': 13.655871629714966}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 54.688621282577515}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 56.92286920547485}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 57.92738223075867}}}}


#### Mga validation
- `name`:
    - Ang pangalan ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Ang isang classifier na may parehong pangalan ay dapat na mayroon na sa shared data directory.
### 5. Predict

Ginagamit ang `predict` action para makakuha ng mga hard at soft prediction (mga probabilidad).

#### Mga input
|     Pangalan    |    Uri     | Paglalarawan |   Kinakailangan  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Ang pangalan ng action. Dapat ay `predict`. | Oo |
| `name`      | `str`    | Ang pangalan ng classifier na gagamitin. | Oo |
| `X`         | `array` o `list` o `str` | Ang test data. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `options["out"]` | `str` | Ang output JSON filename para i-save ang mga hula sa shared data directory. Kung hindi ibinigay, ibabalik ang mga hula sa resulta ng trabaho. | Hindi |

#### Paggamit

In [7]:
job = singularity.run(
    action="predict",
    name="my_classifier",
    X="X_test.npy",  # you do not need to specify the tar extension
)

result = job.result()

print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results):", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results):", result["data"]["probabilities"][:5]
)

Action result status:  ok
Action result message:  Classifier predicted.
Predictions (first five results): [0, 0, 1, 0, 1]
Probabilities (first five results): [[1.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 0.0], [0.0, 1.0]]


#### Mga validation
- `name`:
    - Ang pangalan ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Ang isang classifier na may parehong pangalan ay dapat na mayroon na sa shared data directory.
- `options["out"]`:
    - Ang filename ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Dapat may extension na `.json`.
### 6. Fit-predict

Sine-train ng `fit_predict` action ang isang classifier gamit ang training data at pagkatapos ay ginagamit ito para makakuha ng mga hard at soft prediction (mga probabilidad).

#### Mga input
|     Pangalan    |    Uri     | Paglalarawan |   Kinakailangan  |
|-------------|-------------|-------------|-------------|
| `action`    | `str`    | Ang pangalan ng action. Dapat ay `fit_predict`. | Oo |
| `name`      | `str`    | Ang pangalan ng classifier na gagamitin. | Oo |
| `X_train`   | `array` o `list` o `str` | Ang training data. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `y_train`   | `array` o `list` o `str` | Ang mga target na halaga ng training. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `X_test`    | `array` o `list` o `str` | Ang test data. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `fit_params`| `dictionary`| Mga karagdagang parameter na ipapasa sa `fit` na paraan ng classifier. | Hindi |
| `options["out"]` | `str` | Ang output JSON filename para i-save ang mga hula sa shared data directory. Kung hindi ibinigay, ibabalik ang mga hula sa resulta ng trabaho. | Hindi |

#### Paggamit

In [8]:
job = singularity.run(
    action="delete",
    name="my_classifier",
)

# or you can delete from the shared data directory
# catalog.file_delete("my_classifier.pkl.tar", singularity)

print(job.result())

{'status': 'ok', 'message': 'Classifier deleted.', 'data': {}, 'metadata': {'resource_usage': {}}}


#### Mga validation
- `name`:
    - Ang pangalan ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Ang isang classifier na may parehong pangalan ay dapat na mayroon na sa shared data directory.

- `options["out"]`:
    - Ang filename ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Dapat may extension na `.json`.
### 7. Create-fit-predict

Gumagawa ang `create_fit_predict` action ng isang classifier, sine-train ito gamit ang ibinigay na training data, at pagkatapos ay ginagamit ito para makakuha ng mga hard at soft prediction (mga probabilidad).

#### Mga input
| Pangalan | Uri | Paglalarawan | Kinakailangan |
|------|------|-------------|-----------|
| `action` | `str` | Ang pangalan ng action mula sa `create`, `list`, `fit`, `predict`, `fit_predict`, `create_fit_predict` at `delete`. | Oo |
| `name` | `str` | Ang pangalan ng classifier na gagamitin. | Oo |
| `quantum_classifier` | `str` | Ang uri ng classifier, i.e., `QuantumEnhancedEnsembleClassifier`. Ang default ay `QuantumEnhancedEnsembleClassifier`. | Hindi |
| `X_train` | `array` o `list` o `str` | Ang training data. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `y_train` | `array` o `list` o `str` | Ang mga target na halaga ng training. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `X_test` | `array` o `list` o `str` | Ang test data. Maaaring ito ay isang NumPy array, isang list, o isang string na tumutukoy sa isang filename sa shared data directory. | Oo |
| `fit_params` | `dictionary` | Mga karagdagang parameter na ipapasa sa `fit` na paraan ng classifier. | Hindi |
| `options["save"]` | `boolean` | Kung ise-save ang na-train na classifier sa shared data directory. Ang default ay `True`. | Hindi |
| `options["out"]` | `str` | Ang output JSON filename para i-save ang mga hula sa shared data directory. Kung hindi ibinigay, ibabalik ang mga hula sa resulta ng trabaho. | Hindi |

#### Paggamit

In [9]:
# delete the numpy files from your local disk
os.remove("X_train.npy")
os.remove("y_train.npy")
os.remove("X_test.npy")
os.remove("y_test.npy")

# delete the tar files from your local disk
os.remove("X_train.npy.tar")
os.remove("y_train.npy.tar")
os.remove("X_test.npy.tar")
os.remove("y_test.npy.tar")

# delete the tar files from the shared data
catalog.file_delete("X_train.npy.tar", singularity)
catalog.file_delete("y_train.npy.tar", singularity)
catalog.file_delete("X_test.npy.tar", singularity)
catalog.file_delete("y_test.npy.tar", singularity)

'Requested file was deleted.'

#### Mga validation
- `name`:
    - Kung ang `options["save"]` ay itinakda sa `True`:
        - Ang pangalan ay dapat natatangi, isang string na hanggang 64 character ang haba.
        - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
        - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
        - Walang classifier na may parehong pangalan ang dapat na mayroon na sa shared data directory.

- `options["out"]`:
    - Ang filename ay dapat natatangi, isang string na hanggang 64 character ang haba.
    - Maaari lamang itong maglaman ng alphanumeric na character at underscore.
    - Dapat magsimula sa isang titik at hindi maaaring magtapos sa underscore.
    - Dapat may extension na `.json`.
---

## Magsimula
Mag-authenticate gamit ang iyong [IBM Quantum Platform API key](http://quantum.cloud.ibm.com/), at piliin ang Qiskit Function tulad ng sumusunod:

In [10]:
# Import QiskitFunctionsCatalog to load the
# "Singularity Machine Learning - Classification" function by Multiverse Computing
from qiskit_ibm_catalog import QiskitFunctionsCatalog

# Import the make_moons and the train_test_split functions from scikit-learn
# to create a synthetic dataset and split it into training and test datasets
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

# authentication
# If you have not previously saved your credentials, follow instructions at
# /docs/guides/functions
# to authenticate with your API key.
catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# load "Singularity Machine Learning - Classification" function by Multiverse Computing
singularity = catalog.load("multiverse/singularity")

# generate the synthetic dataset
X, y = make_moons(n_samples=1000)

# split the data into training and test datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

job = singularity.run(
    action="create_fit_predict",
    num_learners=10,
    regularization=0.01,
    optimizer_options={"simulator": True},
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    options={"save": False},
)

# get job status and result
status = job.status()
result = job.result()

print("Job status: ", status)
print("Action result status: ", result["status"])
print("Action result message: ", result["message"])
print("Predictions (first five results): ", result["data"]["predictions"][:5])
print(
    "Probabilities (first five results): ",
    result["data"]["probabilities"][:5],
)
print("Usage metadata: ", result["metadata"]["resource_usage"])

Job status:  QUEUED
Action result status:  ok
Action result message:  Classifier created, fitted, and predicted.
Predictions (first five results):  [0, 0, 1, 0, 0]
Probabilities (first five results):  [[0.87119766531518, 0.1288023346848197], [0.87119766531518, 0.1288023346848197], [0.24470328446479797, 0.7552967155352032], [0.820524432250189, 0.17947556774981072], [0.6847610293419495, 0.31523897065805173]]
Usage metadata:  {'RUNNING: MAPPING': {'CPU_TIME': 10.967791318893433}, 'RUNNING: WAITING_QPU': {'CPU_TIME': 59.91712307929993}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 59.097386837005615}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 56.93338203430176}}


## Halimbawa
Sa halimbawang ito, gagamitin mo ang function na "Singularity Machine Learning - Classification" para ma-classify ang isang dataset na binubuo ng dalawang magkasalubong na kalahating bilog na hugis-buwan. Ang dataset ay synthetic, dalawang-dimensional, at may mga binary na label. Ginawa ito upang maging mahirap para sa mga algorithm tulad ng centroid-based clustering at linear classification.
![Moons dataset](../docs/images/guides/multiverse-computing-singularity/moon_shaped.avif)
Sa prosesong ito, matututunan mo kung paano gumawa ng classifier, i-fit ito sa training data, gamitin ito para mag-predict sa test data, at tanggalin ang classifier pagkatapos.
Bago magsimula, kailangan mong i-install ang [scikit-learn](https://scikit-learn.org/). I-install ito gamit ang sumusunod na command: